In [ ]:
from pathlib import Path
import pandas as pd
from loguru import logger
import requests

start_year = 2018
end_year = 2025

# 1. Sube un nivel desde la carpeta 'notebooks' para llegar a la raíz del proyecto
root = Path().resolve().parent
output_path = root / "data/raw/MunicipiosSequia.xlsx"

# 2. Asegura que la carpeta exista y descarga directamente
output_path.parent.mkdir(parents=True, exist_ok=True)
url = "https://smn.conagua.gob.mx/tools/RESOURCES/Monitor de Sequia en Mexico/MunicipiosSequia.xlsx"

try:
    output_path.write_bytes(requests.get(url).content)
    logger.success(f"¡Archivo descargado y guardado con éxito en: {output_path}!")
except requests.exceptions.RequestException as e:
    logger.error(f"[ERROR DE RED] No se pudo descargar el archivo: {e}")
    raise
except Exception as e:
    logger.error(f"[ERROR INESPERADO] Ocurrió un error al guardar el archivo: {e}")
    raise
    

# 1. Definir rutas desde el notebook (subiendo un nivel a la raíz)
root = Path().resolve().parent
raw_path = root / "data/raw/MunicipiosSequia.xlsx"
interim_dir = root / "data/interim"
interim_dir.mkdir(parents=True, exist_ok=True)
interim_path = interim_dir / "sonora_sequia_tidy.csv"

logger.info(f"Leyendo el archivo crudo de CONAGUA desde {raw_path}...")
df = pd.read_excel(raw_path, engine="openpyxl")

# 1. Filtrar Sonora de forma segura (usando clave que empiece con '26')
cve_col = next((c for c in df.columns if "CVE" in str(c).upper()), None)
if cve_col:
    df_sonora = df[df[cve_col].astype(str).str.zfill(5).str.startswith("26")].copy()
elif "Entidad" in df.columns:
    df_sonora = df[df["Entidad"].str.lower() == "sonora"].copy()
else:
    df_sonora = df.copy()

# 2. Identificar columnas fijas (metadatos) vs columnas de fechas (quincenas)
id_vars_candidates = [
    "Cve_Entidad",
    "Entidad",
    "Cve_Municipio",
    "Municipio",
    "CVE_CONCATENADA",
    "CVE_ENT",
    "CVE_MUN",
    "MUNICIPIO",
    "Abrevia",
]
id_vars = [c for c in df_sonora.columns if c in id_vars_candidates]

# 3. Aplicar melt para pasar a formato largo (Tidy Data)
logger.info("Transformando a formato largo (Tidy Data)...")
df_melted = df_sonora.melt(
    id_vars=id_vars, var_name="fecha", value_name="categoria_sequia"
)

# Limpiar y convertir fechas
df_melted["fecha"] = pd.to_datetime(df_melted["fecha"], errors="coerce")
df_melted = df_melted.dropna(subset=["fecha"])

# 4. Filtrar estrictamente el rango de años
df_melted["Anio"] = df_melted["fecha"].dt.year
df_filtered = df_melted[
    (df_melted["Anio"] >= start_year) & (df_melted["Anio"] <= end_year)
].copy()

# 5. Añadir escala numérica de severidad
severity_map = {"Sin Sequía": 0, "D0": 1, "D1": 2, "D2": 3, "D3": 4, "D4": 5}
df_filtered["severidad_num"] = (
    df_filtered["categoria_sequia"].map(severity_map).fillna(0).astype(int)
)

# 6. Guardar un archivo CSV separado por cada año
for year, df_year in df_filtered.groupby("Anio"):
    interim_path = interim_dir / f"sonora_sequia_{year}.csv"
    df_year.to_csv(interim_path, index=False, encoding="utf-8-sig")
    logger.success(
        f"Guardado año {year} -> {interim_path.name} ({len(df_year)} registros)"
    )

logger.success("¡Proceso de sequía por año completado exitosamente!")

2026-09-24 23:11:31.069 | INFO     | __main__:<module>:28 - Leyendo el archivo crudo de CONAGUA desde C:\Users\patyq\OneDrive - Universidad de Sonora\MDC\IngCaracteristicas\sonora-agriclimate-eda\data\raw\MunicipiosSequia.xlsx...


¡Archivo descargado y guardado en: C:\Users\patyq\OneDrive - Universidad de Sonora\MDC\IngCaracteristicas\sonora-agriclimate-eda\data\raw\MunicipiosSequia.xlsx!


2026-09-24 23:11:36.342 | INFO     | __main__:<module>:55 - Transformando a formato largo (Tidy Data)...
C:\Users\patyq\AppData\Local\Temp\ipykernel_12092\203040951.py:61: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_melted["fecha"] = pd.to_datetime(df_melted["fecha"], errors="coerce")
2026-09-24 23:11:36.432 | SUCCESS  | __main__:<module>:80 - Guardado año 2018 -> sonora_sequia_2018.csv (1728 registros)
2026-09-24 23:11:36.438 | SUCCESS  | __main__:<module>:80 - Guardado año 2019 -> sonora_sequia_2019.csv (1728 registros)
2026-09-24 23:11:36.444 | SUCCESS  | __main__:<module>:80 - Guardado año 2020 -> sonora_sequia_2020.csv (1728 registros)
2026-09-24 23:11:36.453 | SUCCESS  | __main__:<module>:80 - Guardado año 2021 -> sonora_sequia_2021.csv (1728 registros)
2026-09-24 23:11:36.459 | SUCCESS  | __main__:<module>:80 - Guardado año 2022 -> sono